# Notatnik do podzialu danych 

## Załadowanie bibliotek


In [30]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from pathlib import Path

## Załadowanie danych 

In [7]:
df_f = pd.read_csv('../data/processed/main/df_merged.csv', index_col = 0)
df_s = pd.read_csv('../data/processed/main/df_severity.csv', index_col = 0)
df_p = pd.read_csv('../data/processed/main/df_pure_premium.csv', index_col = 0)


In [8]:
df_freq = df_f.copy()
df_sev = df_s.copy()
df_pp = df_p.copy()

## Podział danych

### Stworzenie zmiennej do stratyfikacji

In [14]:
df_freq['HasClaim'] = (df_freq['ClaimNb'] > 0).astype(int)
df_freq[df_freq['HasClaim'] > 0]


,PolicyID,ClaimNb,Exposure,Power,CarAge,DriverAge,Brand,Gas,Region,Density,TotalClaimAmount,ClaimCount,HasClaim
32,33,1,0.75,g,1,61,Japanese (except Nissan) or Korean,Regular,Ile-de-France,27000,302.0,1.0,1
40,41,1,0.14,l,5,50,Japanese (except Nissan) or Korean,Diesel,Basse-Normandie,56,2001.0,1.0,1
91,92,1,0.14,d,0,36,Japanese (except Nissan) or Korean,Regular,Ile-de-France,4792,1449.0,1.0,1
95,96,2,0.62,j,0,51,Japanese (except Nissan) or Korean,Regular,Ile-de-France,27000,10870.0,2.0,1
141,142,1,0.75,e,0,34,Japanese (except Nissan) or Korean,Regular,Nord-Pas-de-Calais,1565,1390.0,1.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
412261,412262,1,0.42,g,1,72,Japanese (except Nissan) or Korean,Diesel,Aquitaine,123,1426.0,1.0,1
412269,412270,1,0.36,g,1,53,Japanese (except Nissan) or Korean,Diesel,Nord-Pas-de-Calais,4540,1364.0,1.0,1
412663,412664,1,0.22,h,0,34,Japanese (except Nissan) or Korean,Regular,Ile-de-France,6570,7603.0,1.0,1
412925,412926,1,0.14,g,2,51,Japanese (except Nissan) or Korean,Diesel,Aquitaine,178,1751.0,1.0,1


### Podzial - train (70%), validation (15%), test (15%)

Podzial ramki danych df_freq

In [15]:
# Train: 70%, pozostałe: 30%
train_df_freq, temp_df = train_test_split(
    df_freq,
    test_size=0.30,
    random_state=42,
    stratify=df_freq['HasClaim']
)

# Validation: 15%, Test: 15%
val_df_freq, test_df_freq = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df['HasClaim']
)

Sprawdzenie podzialu

In [16]:
splits = {
    'Train': train_df_freq,
    'Validation': val_df_freq,
    'Test': test_df_freq
}

for name, df in splits.items():
    print(
        f"{name}: {len(df):,} observations "
        f"({len(df) / len(df_freq):.2%}), "
        f"claims: {(df['ClaimNb'] > 0).sum():,} "
        f"({(df['ClaimNb'] > 0).mean():.2%})"
    )

Train: 289,218 observations (70.00%), claims: 10,773 (3.72%)
Validation: 61,975 observations (15.00%), claims: 2,308 (3.72%)
Test: 61,976 observations (15.00%), claims: 2,309 (3.73%)


Podzial wyglada w porzadku - struktura czestotliwosci jest zachowana

### Stworzenie pozostalych zbiorow dla severity i pure premium na podstawie uzyskanego wyzej podzialu

In [17]:
train_ids = set(train_df_freq['PolicyID'])
val_ids = set(val_df_freq['PolicyID'])
test_ids = set(test_df_freq['PolicyID'])

# Severity
severity_train = df_sev[
    df_sev['PolicyID'].isin(train_ids)
].copy()

severity_val = df_sev[
    df_sev['PolicyID'].isin(val_ids)
].copy()

severity_test = df_sev[
    df_sev['PolicyID'].isin(test_ids)
].copy()

# Pure Premium
pp_train = df_pp[
    df_pp['PolicyID'].isin(train_ids)
].copy()

pp_val = df_pp[
    df_pp['PolicyID'].isin(val_ids)
].copy()

pp_test = df_pp[
    df_pp['PolicyID'].isin(test_ids)
].copy()

Kontrola podzialu pure premium

In [28]:
pure_premium_splits = {
    'Train': pp_train,
    'Validation': pp_val,
    'Test': pp_test
}

for name, df in pure_premium_splits.items():
    print('*' * 100)
    print(
        f"{name}: {len(df):,} observations "
        f"({len(df) / len(df_freq):.2%}), "
        f"claims: {(df['ClaimNb'] > 0).sum():,} "
        f"({(df['ClaimNb'] > 0).mean():.2%}) "
        f"mean pure premium: {df['PurePremium'].mean()}"
    )

****************************************************************************************************
Train: 289,218 observations (70.00%), claims: 10,773 (3.72%) mean pure premium: 393.1337210132063
****************************************************************************************************
Validation: 61,975 observations (15.00%), claims: 2,308 (3.72%) mean pure premium: 298.4609674220843
****************************************************************************************************
Test: 61,976 observations (15.00%), claims: 2,309 (3.73%) mean pure premium: 341.2169280828993


In [36]:
pp_stats = pd.DataFrame({
    'Train': np.percentile(pp_train['PurePremium'], [90, 95, 99, 99.9]),
    'Validation': np.percentile(pp_val['PurePremium'], [90, 95, 99, 99.9]),
    'Test': np.percentile(pp_test['PurePremium'], [90, 95, 99, 99.9])
})

pp_stats

,Train,Validation,Test
0,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.000000
2,2932.627689,2640.292857,2929.385965
3,34605.341667,39168.833333,32761.729167


Kontrola podzialu severity

In [20]:
severity_splits = {
    'Train': severity_train,
    'Validation': severity_val,
    'Test': severity_test
}

total_severity = len(df_sev)

for name, df in severity_splits.items():
    print(
        f"{name}: {len(df):,} observations "
        f"({len(df) / total_severity:.2%}) "
        f"mean severity: {df['Severity'].mean()}"
    )

Train: 10,773 observations (70.00%) mean severity: 2049.0974426807757
Validation: 2,308 observations (15.00%) mean severity: 2520.4094454072792
Test: 2,309 observations (15.00%) mean severity: 1835.7560993214959


Sprawdzenie rozkladu wartosci severity

In [21]:
severity_stats = pd.DataFrame({
    'Train': severity_train['Severity'].describe(),
    'Validation': severity_val['Severity'].describe(),
    'Test': severity_test['Severity'].describe()
})

severity_stats

,Train,Validation,Test
count,1.077300e+04,2308.000000,2309.000000
mean,2.049097e+03,2520.409445,1835.756099
std,2.102209e+04,18614.004804,6079.914025
min,2.000000e+00,2.000000,4.000000
25%,7.210000e+02,697.875000,763.000000
50%,1.155000e+03,1154.500000,1161.000000
75%,1.244000e+03,1239.000000,1261.000000
max,2.036833e+06,701726.000000,137003.000000


## Zapisanie podzielonych danych do plikow parquet

In [24]:


splits_dir = Path('../data/processed')

datasets = {
    'train/frequency': train_df_freq,
    'val/frequency': val_df_freq,
    'test/frequency': test_df_freq,
    'train/severity': severity_train,
    'val/severity': severity_val,
    'test/severity': severity_test,
    'train/pure_premium': pp_train,
    'val/pure_premium': pp_val,
    'test/pure_premium': pp_test
}

for name, df in datasets.items():
    path = splits_dir / name
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path.with_suffix('.parquet'), index=False)